# Diffusion Model Training

Two-stage training for AMP latent diffusion: unconditional Phase 1, then organism-conditional Phase 2 (with or without potency-level embeddings).


### Project paths

This notebook lives in `notebooks/`. Shared code is under `src/`, data under `data/`.
On Colab, set `ROOT` to your cloned repo (or Drive copy) instead of `Path('..')`.


In [ ]:
import sys
from pathlib import Path

# Repo root (parent of notebooks/). On Colab, replace with your clone path, e.g.:
# ROOT = Path('/content/drive/MyDrive/your-thesis-repo')
ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.paths import DATA_DIR, CHECKPOINTS_DIR, RESULTS_DIR, PROJECT_ROOT
print('PROJECT_ROOT :', PROJECT_ROOT)
print('DATA_DIR     :', DATA_DIR)
print('CHECKPOINTS  :', CHECKPOINTS_DIR)
print('RESULTS      :', RESULTS_DIR)


## Environment setup

Install Weights & Biases for experiment logging.


In [ ]:
!pip install -q wandb

## Phase 1 - Unconditional diffusion

Trains the base diffusion model on peptide latents with no organism or potency conditioning.

After Phase 1 finishes, this same cell continues into a Phase 2 (organism-conditional, **no** potency levels). For a separate Phase 2-only run, use the dedicated Phase 2 cells below instead.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, random_split
)
from tqdm import tqdm
from transformers.models.bert.modeling_bert import BertEncoder, BertConfig

print("Imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")


import wandb

WANDB_PROJECT = "AMP_Generation"
WANDB_ENTITY  = None

class _SmoothBuffer:
    def __init__(self, size=50):
        self._buf  = []
        self._size = size

    def push(self, v):
        self._buf.append(v)
        if len(self._buf) > self._size:
            self._buf.pop(0)

    @property
    def mean(self):
        return float(np.mean(self._buf)) if self._buf else float("nan")

    @property
    def std(self):
        return float(np.std(self._buf))  if len(self._buf) > 1 else 0.0


def timestep_embedding(timesteps, dim, max_period=10000):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32)
        / half
    ).to(device=timesteps.device)
    args      = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):
    def __init__(self, num_class=2, max_length=1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels   = 128
        self.num_class      = num_class
        self.drop_out       = 0.1
        self.max_length     = max_length
        config = BertConfig()
        config.hidden_dropout_prob     = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads     = 6
        config.num_hidden_layers       = 3
        config._attn_implementation    = "eager"
        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer(
            "position_ids",
            torch.arange(config.max_position_embeddings).expand((1, -1))
        )
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout   = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x, timesteps, control=None):
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)
        emb_x      = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids    = self.position_ids[:, :seq_length]
        emb_inputs = (
            self.position_embeddings(pos_ids)
            + emb_x
            + emb.unsqueeze(1).expand(-1, seq_length, -1)
        )
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))
        attn_mask  = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


def _extract_into_tensor(arr, timesteps, broadcast_shape):
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


def mean_flat(tensor):
    return tensor.mean(dim=list(range(1, len(tensor.shape))))


class MyDiffusion:
    def __init__(self, num_timesteps, betas):
        self.betas         = betas
        alphas             = 1.0 - betas
        self.num_timesteps = num_timesteps
        self.alphas_cumprod      = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod           = np.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev)
            * np.sqrt(alphas)
            / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        return (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + _extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
            * noise
        )

    def _scale_timesteps(self, t):
        return t.float() * (1000.0 / self.num_timesteps)

    def get_x_start(self, x_start_mean, std):
        return x_start_mean + std * torch.randn_like(x_start_mean)

    def cal_loss(self, model, x_0, t, condition=None):
        std = _extract_into_tensor(
            self.sqrt_one_minus_alphas_cumprod,
            torch.tensor([0]).to(x_0.device),
            x_0.shape,
        )
        x_start      = self.get_x_start(x_0, std)
        noise        = torch.randn_like(x_0)
        x_t          = self.q_sample(x_start, t, noise=noise)
        model_output = model(x_t, self._scale_timesteps(t), condition)
        target       = x_start
        mse_loss = mean_flat((target - model_output) ** 2)
        t0_mask  = (t == 0)
        t0_loss  = mean_flat((x_0 - model_output) ** 2)
        return torch.where(t0_mask, t0_loss, mse_loss)

    def q_posterior_mean_variance(self, x_start, x_t, t):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        var     = _extract_into_tensor(self.posterior_variance,             t, x_t.shape)
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, var, log_var

    def p_mean_variance(self, model, x, t, clip_denoised=True, cond=None):
        model_output   = model(x, self._scale_timesteps(t), cond)
        pred_xstart    = model_output.clamp(-1, 1) if clip_denoised else model_output
        model_mean, _, _ = self.q_posterior_mean_variance(pred_xstart, x, t)
        model_log_var  = _extract_into_tensor(
            self.posterior_log_variance_clipped, t, x.shape
        )
        return {"mean": model_mean, "log_variance": model_log_var, "pred_xstart": pred_xstart}

    def p_sample(self, model, x, t, clip_denoised=False, cond=None):
        out          = self.p_mean_variance(model, x, t, clip_denoised, cond)
        noise        = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        sample       = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}

    def p_sample_loop(self, model, shape, cond=None, device="cuda"):
        img = torch.randn(*shape, device=device)
        for i in list(range(self.num_timesteps))[::-1]:
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                out = self.p_sample(model, img, t, clip_denoised=False, cond=cond)
                img = out["sample"]
        return img


def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps=500):
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)


class UniformSampler:
    def __init__(self, num_timesteps=500):
        self._weights = np.ones([num_timesteps])

    def sample(self, batch_size, device):
        p          = self._weights / np.sum(self._weights)
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices    = torch.from_numpy(indices_np).long().to(device)
        weights_np = 1 / (len(p) * p[indices_np])
        weights    = torch.from_numpy(weights_np).float().to(device)
        return indices, weights


class LatentDataset_nocond(Dataset):
    def __init__(self, npy_path):
        raw = np.load(npy_path).astype(np.float32)
        if raw.ndim == 3:
            raw = raw.squeeze(1)
        raw = raw * 25.0
        self.data = raw[:, np.newaxis, :]
        print(f"[nocond] {len(self.data)} samples  shape={self.data.shape[1:]}")

    def __len__(self):  return len(self.data)
    def __getitem__(self, idx): return torch.from_numpy(self.data[idx])


class LatentDataset_cond(Dataset):
    def __init__(self, latents_path, organisms_path):
        latents   = np.load(latents_path).astype(np.float32)
        organisms = np.load(organisms_path).astype(np.int64)
        if latents.ndim == 3:
            latents = latents.squeeze(1)
        assert len(latents) == len(organisms), (
            f"Length mismatch: {len(latents)} latents vs {len(organisms)} labels"
        )
        latents = latents * 25.0
        self.latents       = latents[:, np.newaxis, :]
        self.organisms     = organisms
        self.num_organisms = int(organisms.max()) + 1
        print(f"[cond] {len(self.latents)} samples  organisms={self.num_organisms}")
        for i in range(self.num_organisms):
            print(f"  org {i}: {(self.organisms == i).sum()} samples")

    def __len__(self):  return len(self.latents)
    def __getitem__(self, idx):
        return (
            torch.from_numpy(self.latents[idx]),
            torch.tensor(self.organisms[idx], dtype=torch.long),
        )


def make_weighted_sampler(dataset):
    organisms    = dataset.organisms
    num_orgs     = dataset.num_organisms
    counts       = np.bincount(organisms, minlength=num_orgs).astype(np.float32)
    class_weight = 1.0 / np.where(counts > 0, counts, 1.0)
    sample_weight = class_weight[organisms]
    return WeightedRandomSampler(
        weights=torch.from_numpy(sample_weight).float(),
        num_samples=len(sample_weight),
        replacement=True,
    )


print("Definitions loaded.")


P1 = dict(
    train_latents = f"{INPUT_PATH}/latents_p1/train_latents.npy",
    val_latents   = f"{INPUT_PATH}/latents_p1/val_latents.npy",
    save_path     = f"{INPUT_PATH}/checkpoints_diffusion/phase1",
    lr            = 1e-4,
    num_epochs    = 100,
    batch_size    = 64,
    num_steps     = 500,
    num_workers   = 2,
    save_every    = 10,
)

os.makedirs(P1["save_path"], exist_ok=True)
os.makedirs(f"{P1['save_path']}/model", exist_ok=True)

run_p1 = wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = "phase1_unconditional",
    group   = "AMP_diffusion",
    tags    = ["phase1", "unconditional"],
    config  = {
        **P1,
        "phase":        1,
        "model":        "DModel-BertEncoder",
        "num_class":    2,
        "max_length":   1,
        "device":       str(device),
        "optimizer":    "AdamW",
    },
    reinit  = True,
)

train_data_p1   = LatentDataset_nocond(P1["train_latents"])
val_data_p1     = LatentDataset_nocond(P1["val_latents"])
train_loader_p1 = DataLoader(
    train_data_p1, batch_size=P1["batch_size"],
    shuffle=True,  num_workers=P1["num_workers"], pin_memory=True, drop_last=False,
)
val_loader_p1 = DataLoader(
    val_data_p1, batch_size=P1["batch_size"] // 10,
    shuffle=False, num_workers=P1["num_workers"], pin_memory=True, drop_last=False,
)
print(f"P1  Train={len(train_data_p1)}  Val={len(val_data_p1)}")

model_p1 = DModel(num_class=2, max_length=1).to(device)
diff_p1  = create_gaussian_diffusion(P1["num_steps"])
samp_p1  = UniformSampler(P1["num_steps"])
opt_p1   = AdamW(model_p1.parameters(), lr=P1["lr"])

wandb.watch(model_p1, log="gradients", log_freq=200)

log_path_p1 = f"{P1['save_path']}/train.log"
if not os.path.exists(log_path_p1):
    with open(log_path_p1, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")

best_loss_p1  = float("inf")
p1_global_step = 0
p1_smooth      = _SmoothBuffer(size=50)
p1_epoch_batch_losses = []

for epoch in range(P1["num_epochs"]):
    model_p1.train()
    train_loss, n_train   = 0.0, 0
    start                 = time.time()
    log_rows              = []
    epoch_batch_losses_p1 = []

    for i, data in enumerate(tqdm(train_loader_p1, desc=f"[P1 Ep {epoch:3d}] train")):
        data = data.to(device)
        opt_p1.zero_grad()
        t, _ = samp_p1.sample(data.shape[0], device)
        loss  = diff_p1.cal_loss(model_p1, data, t, condition=None).mean()
        loss.backward()

        grad_norm = float(
            torch.nn.utils.clip_grad_norm_(model_p1.parameters(), max_norm=1e9)
        )

        opt_p1.step()
        loss_val = loss.item()
        train_loss += loss_val
        n_train    += 1

        p1_smooth.push(loss_val)
        epoch_batch_losses_p1.append(loss_val)

        log_rows.append(
            f"{epoch},{i},train,{loss_val:.6f},{time.time()-start:.2f}"
        )

        wandb.log({
            "p1/batch/loss_raw":        loss_val,
            "p1/batch/loss_smoothed":   p1_smooth.mean,
            "p1/batch/loss_std":        p1_smooth.std,
            "p1/batch/grad_norm":       grad_norm,
            "p1/batch/lr":              opt_p1.param_groups[0]["lr"],
            "p1/step":                  p1_global_step,
        }, step=p1_global_step)

        p1_global_step += 1

    train_loss /= n_train

    model_p1.eval()
    val_loss, n_val   = 0.0, 0
    val_batch_losses  = []

    with torch.no_grad():
        for i, vdata in enumerate(tqdm(val_loader_p1, desc=f"[P1 Ep {epoch:3d}] val  ")):
            vdata = vdata.to(device)
            t, _  = samp_p1.sample(vdata.shape[0], device)
            loss  = diff_p1.cal_loss(model_p1, vdata, t, condition=None).mean()
            val_loss += loss.item()
            val_batch_losses.append(loss.item())
            n_val    += 1
            log_rows.append(
                f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}"
            )

    val_loss /= n_val
    epoch_time = time.time() - start

    with open(log_path_p1, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    gen_gap = val_loss - train_loss
    wandb.log({
        "p1/epoch/train_loss":          train_loss,
        "p1/epoch/val_loss":            val_loss,
        "p1/epoch/generalization_gap":  gen_gap,
        "p1/epoch/epoch_time_s":        epoch_time,
        "p1/epoch/lr":                  opt_p1.param_groups[0]["lr"],
        "p1/epoch/train_loss_hist":     wandb.Histogram(epoch_batch_losses_p1),
        "p1/epoch/val_loss_hist":       wandb.Histogram(val_batch_losses),
        "p1/epoch":                     epoch,
        "p1/step":                      p1_global_step,
    }, step=p1_global_step)

    if val_loss < best_loss_p1:
        best_loss_p1 = val_loss
        ckpt_path    = f"{P1['save_path']}/best_model.pth"
        torch.save(model_p1.state_dict(), ckpt_path)
        wandb.run.summary["p1/best_val_loss"]  = best_loss_p1
        wandb.run.summary["p1/best_val_epoch"] = epoch
        print(f"  ✓ Best Phase 1 model saved  val={val_loss:.5f}")

    if (epoch + 1) % P1["save_every"] == 0:
        ckpt = (f"{P1['save_path']}/model/"
                f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth")
        torch.save(model_p1.state_dict(), ckpt)

    print(f"[P1] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
          f"| gap {gen_gap:+.5f} | {epoch_time:.1f}s")

wandb.finish()
print("Phase 1 complete.")

P2 = dict(
    latents_path   = f"{INPUT_PATH}/latent_two_organism/cond_latents_new/cond_latents.npy",
    organisms_path = f"{INPUT_PATH}/latent_two_organism/cond_latents_new/cond_organisms.npy",
    phase1_ckpt    = f"{INPUT_PATH}/checkpoints_diffusion/phase1/best_model.pth",
    save_path      = f"{INPUT_PATH}/checkpoints_diffusion/phase2_2org",
    lr             = 1e-4,
    num_epochs     = 300,
    batch_size     = 512,
    num_steps      = 500,
    num_workers    = 2,
    save_every     = 10,
    cfg_dropout    = 0.10,
)

os.makedirs(P2["save_path"], exist_ok=True)
os.makedirs(f"{P2['save_path']}/model", exist_ok=True)

all_data_p2   = LatentDataset_cond(P2["latents_path"], P2["organisms_path"])
num_organisms = all_data_p2.num_organisms
NULL_LABEL    = num_organisms

length     = len(all_data_p2)
train_size = int(0.8 * length)
val_size   = length - train_size
train_data_p2, val_data_p2 = random_split(
    all_data_p2, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_organisms = all_data_p2.organisms[list(train_data_p2.indices)]
counts     = np.bincount(train_organisms, minlength=num_organisms).astype(np.float32)
class_wt   = 1.0 / np.where(counts > 0, counts, 1.0)
sample_wts = class_wt[train_organisms]
weighted_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_wts).float(),
    num_samples=len(sample_wts),
    replacement=True,
)

train_loader_p2 = DataLoader(
    train_data_p2, batch_size=P2["batch_size"],
    sampler=weighted_sampler,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=True,
)
val_loader_p2 = DataLoader(
    val_data_p2, batch_size=P2["batch_size"] // 10,
    shuffle=False,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=False,
)
print(f"\nP2  Train={train_size}  Val={val_size}  "
      f"Organisms={num_organisms}  NullLabel={NULL_LABEL}")

run_p2 = wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = "phase2_CFG_conditional",
    group   = "AMP_diffusion",
    tags    = ["phase2", "conditional", "CFG"],
    config  = {
        **P2,
        "phase":         2,
        "model":         "DModel-BertEncoder",
        "num_organisms": num_organisms,
        "null_label":    NULL_LABEL,
        "device":        str(device),
        "optimizer":     "AdamW",
    },
    reinit  = True,
)

model_p2 = DModel(num_class=num_organisms + 1, max_length=1).to(device)

if os.path.exists(P2["phase1_ckpt"]):
    p1_ckpt = torch.load(P2["phase1_ckpt"], map_location=device)
    if isinstance(p1_ckpt, dict) and "model_state" in p1_ckpt:
        p1_ckpt = p1_ckpt["model_state"]
    p2_state = model_p2.state_dict()
    to_load  = {k: v for k, v in p1_ckpt.items()
                if k in p2_state and v.shape == p2_state[k].shape}
    skipped  = [k for k in p1_ckpt if k not in to_load]
    model_p2.load_state_dict(to_load, strict=False)
    n_loaded = len(to_load)
    print(f"Loaded {n_loaded} tensors from Phase 1. Skipped: {skipped}")
    wandb.run.summary["p2/p1_tensors_loaded"] = n_loaded
    wandb.run.summary["p2/p1_tensors_skipped"] = len(skipped)
else:
    print("Phase 1 checkpoint not found.")

diff_p2 = create_gaussian_diffusion(P2["num_steps"])
samp_p2 = UniformSampler(P2["num_steps"])
opt_p2  = AdamW(model_p2.parameters(), lr=P2["lr"])

wandb.watch(model_p2, log="gradients", log_freq=200)

log_path_p2 = f"{P2['save_path']}/train.log"
if not os.path.exists(log_path_p2):
    with open(log_path_p2, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")

best_loss_p2   = float("inf")
cfg_dropout    = P2["cfg_dropout"]
p2_global_step = 0
p2_smooth      = _SmoothBuffer(size=50)
p2_smooth_cfg  = _SmoothBuffer(size=50)
p2_smooth_cond = _SmoothBuffer(size=50)

print("\nStarting Phase 2 training...")

for epoch in range(P2["num_epochs"]):
    model_p2.train()
    train_loss, n_train    = 0.0, 0
    cfg_loss_sum,  n_cfg   = 0.0, 0
    cond_loss_sum, n_cond  = 0.0, 0
    start    = time.time()
    log_rows = []
    epoch_batch_losses_p2  = []

    for i, (data, organism_ids) in enumerate(
        tqdm(train_loader_p2, desc=f"[P2 Ep {epoch:3d}] train")
    ):
        data         = data.to(device)
        organism_ids = organism_ids.to(device)
        drop_mask    = torch.rand(organism_ids.shape[0], device=device) < cfg_dropout
        organism_ids = torch.where(
            drop_mask,
            torch.full_like(organism_ids, NULL_LABEL),
            organism_ids,
        )
        opt_p2.zero_grad()
        t, _ = samp_p2.sample(data.shape[0], device)
        loss  = diff_p2.cal_loss(model_p2, data, t, condition=organism_ids).mean()
        loss.backward()

        grad_norm = float(
            torch.nn.utils.clip_grad_norm_(model_p2.parameters(), max_norm=1e9)
        )

        opt_p2.step()
        loss_val    = loss.item()
        n_dropped   = int(drop_mask.sum().item())
        n_kept      = len(drop_mask) - n_dropped
        train_loss += loss_val
        n_train    += 1

        if n_dropped > 0:
            cfg_loss_sum += loss_val; n_cfg += 1
            p2_smooth_cfg.push(loss_val)
        if n_kept > 0:
            cond_loss_sum += loss_val; n_cond += 1
            p2_smooth_cond.push(loss_val)

        p2_smooth.push(loss_val)
        epoch_batch_losses_p2.append(loss_val)

        log_rows.append(f"{epoch},{i},train,{loss_val:.6f},{time.time()-start:.2f}")

        wandb.log({
            "p2/batch/loss_raw":          loss_val,
            "p2/batch/loss_smoothed":     p2_smooth.mean,
            "p2/batch/loss_std":          p2_smooth.std,
            "p2/batch/loss_cfg_smooth":   p2_smooth_cfg.mean,
            "p2/batch/loss_cond_smooth":  p2_smooth_cond.mean,
            "p2/batch/n_cfg_dropped":     n_dropped,
            "p2/batch/n_conditioned":     n_kept,
            "p2/batch/grad_norm":         grad_norm,
            "p2/batch/lr":                opt_p2.param_groups[0]["lr"],
            "p2/step":                    p2_global_step,
        }, step=p2_global_step)

        p2_global_step += 1

    train_loss /= n_train

    model_p2.eval()
    val_loss, n_val  = 0.0, 0
    val_batch_losses = []

    with torch.no_grad():
        for i, (vdata, organism_ids) in enumerate(
            tqdm(val_loader_p2, desc=f"[P2 Ep {epoch:3d}] val  ")
        ):
            vdata        = vdata.to(device)
            organism_ids = organism_ids.to(device)
            t, _  = samp_p2.sample(vdata.shape[0], device)
            loss  = diff_p2.cal_loss(model_p2, vdata, t, condition=organism_ids).mean()
            val_loss += loss.item()
            val_batch_losses.append(loss.item())
            n_val    += 1
            log_rows.append(f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}")

    val_loss /= n_val
    epoch_time = time.time() - start

    with open(log_path_p2, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    gen_gap      = val_loss - train_loss
    avg_cfg_loss  = cfg_loss_sum  / n_cfg  if n_cfg  > 0 else float("nan")
    avg_cond_loss = cond_loss_sum / n_cond if n_cond > 0 else float("nan")

    wandb.log({
        "p2/epoch/train_loss":           train_loss,
        "p2/epoch/val_loss":             val_loss,
        "p2/epoch/generalization_gap":   gen_gap,
        "p2/epoch/cfg_dropout_loss":     avg_cfg_loss,
        "p2/epoch/cond_loss":            avg_cond_loss,
        "p2/epoch/cfg_vs_cond_delta":    avg_cfg_loss - avg_cond_loss
                                         if not (math.isnan(avg_cfg_loss) or
                                                 math.isnan(avg_cond_loss)) else float("nan"),
        "p2/epoch/epoch_time_s":         epoch_time,
        "p2/epoch/lr":                   opt_p2.param_groups[0]["lr"],
        "p2/epoch/train_loss_hist":      wandb.Histogram(epoch_batch_losses_p2),
        "p2/epoch/val_loss_hist":        wandb.Histogram(val_batch_losses),
        "p2/epoch":                      epoch,
        "p2/step":                       p2_global_step,
    }, step=p2_global_step)

    if val_loss < best_loss_p2:
        best_loss_p2 = val_loss
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
            },
            f"{P2['save_path']}/best_model.pth",
        )
        wandb.run.summary["p2/best_val_loss"]  = best_loss_p2
        wandb.run.summary["p2/best_val_epoch"] = epoch
        print(f"  ✓ Best model saved  val={val_loss:.5f}")

    if (epoch + 1) % P2["save_every"] == 0:
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
            },
            f"{P2['save_path']}/model/"
            f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth",
        )

    print(f"[P2] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
          f"| gap {gen_gap:+.5f} | cfg {avg_cfg_loss:.5f} | cond {avg_cond_loss:.5f} "
          f"| {epoch_time:.1f}s")

wandb.finish()
print("Phase 2 training complete.")

## Phase 2 - Without MIC leveling

Organism-conditional diffusion only (CFG). Uses `cond_latents.npy` + `cond_organisms.npy`. **No** potency level embeddings. Starts from the Phase 1 checkpoint.


In [ ]:
# Phase 2 — organism-conditional diffusion (NO potency levels)
#
# Needs:
#   {LATENT_DIR}/cond_latents.npy
#   {LATENT_DIR}/cond_organisms.npy
# Optional warm-start:
#   checkpoints_diffusion/phase1/best_model.pth

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, random_split
)
from tqdm import tqdm
from transformers.models.bert.modeling_bert import BertEncoder, BertConfig

print("Imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")

import wandb

WANDB_PROJECT = "AMP_Generation"
WANDB_ENTITY = None


class _SmoothBuffer:
    def __init__(self, size=50):
        self._buf = []
        self._size = size

    def push(self, v):
        self._buf.append(v)
        if len(self._buf) > self._size:
            self._buf.pop(0)

    @property
    def mean(self):
        return float(np.mean(self._buf)) if self._buf else float("nan")

    @property
    def std(self):
        return float(np.std(self._buf)) if len(self._buf) > 1 else 0.0


def timestep_embedding(timesteps, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32)
        / half
    ).to(device=timesteps.device)
    args = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):
    def __init__(self, num_class=2, max_length=1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels = 128
        self.num_class = num_class
        self.drop_out = 0.1
        self.max_length = max_length
        config = BertConfig()
        config.hidden_dropout_prob = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads = 6
        config.num_hidden_layers = 3
        config._attn_implementation = "eager"
        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer(
            "position_ids",
            torch.arange(config.max_position_embeddings).expand((1, -1)),
        )
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x, timesteps, control=None):
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)
        emb_x = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids = self.position_ids[:, :seq_length]
        emb_inputs = (
            self.position_embeddings(pos_ids)
            + emb_x
            + emb.unsqueeze(1).expand(-1, seq_length, -1)
        )
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))
        attn_mask = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


def _extract_into_tensor(arr, timesteps, broadcast_shape):
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


def mean_flat(tensor):
    return tensor.mean(dim=list(range(1, len(tensor.shape))))


class MyDiffusion:
    def __init__(self, num_timesteps, betas):
        self.betas = betas
        alphas = 1.0 - betas
        self.num_timesteps = num_timesteps
        self.alphas_cumprod = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod = np.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev)
            * np.sqrt(alphas)
            / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        return (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + _extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
            * noise
        )

    def _scale_timesteps(self, t):
        return t.float() * (1000.0 / self.num_timesteps)

    def get_x_start(self, x_start_mean, std):
        return x_start_mean + std * torch.randn_like(x_start_mean)

    def cal_loss(self, model, x_0, t, condition=None):
        std = _extract_into_tensor(
            self.sqrt_one_minus_alphas_cumprod,
            torch.tensor([0]).to(x_0.device),
            x_0.shape,
        )
        x_start = self.get_x_start(x_0, std)
        noise = torch.randn_like(x_0)
        x_t = self.q_sample(x_start, t, noise=noise)
        model_output = model(x_t, self._scale_timesteps(t), control=condition)
        target = x_start
        mse_loss = mean_flat((target - model_output) ** 2)
        t0_mask = t == 0
        t0_loss = mean_flat((x_0 - model_output) ** 2)
        return torch.where(t0_mask, t0_loss, mse_loss)

    def q_posterior_mean_variance(self, x_start, x_t, t):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        var = _extract_into_tensor(self.posterior_variance, t, x_t.shape)
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, var, log_var

    def p_mean_variance(self, model, x, t, clip_denoised=True, cond=None):
        model_output = model(x, self._scale_timesteps(t), cond)
        pred_xstart = model_output.clamp(-1, 1) if clip_denoised else model_output
        model_mean, _, _ = self.q_posterior_mean_variance(pred_xstart, x, t)
        model_log_var = _extract_into_tensor(
            self.posterior_log_variance_clipped, t, x.shape
        )
        return {"mean": model_mean, "log_variance": model_log_var, "pred_xstart": pred_xstart}

    def p_sample(self, model, x, t, clip_denoised=False, cond=None):
        out = self.p_mean_variance(model, x, t, clip_denoised, cond)
        noise = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        sample = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}

    def p_sample_loop(self, model, shape, cond=None, device="cuda"):
        img = torch.randn(*shape, device=device)
        for i in list(range(self.num_timesteps))[::-1]:
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                out = self.p_sample(model, img, t, clip_denoised=False, cond=cond)
                img = out["sample"]
        return img


def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps=500):
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)


class UniformSampler:
    def __init__(self, num_timesteps=500):
        self._weights = np.ones([num_timesteps])

    def sample(self, batch_size, device):
        p = self._weights / np.sum(self._weights)
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices = torch.from_numpy(indices_np).long().to(device)
        weights_np = 1 / (len(p) * p[indices_np])
        weights = torch.from_numpy(weights_np).float().to(device)
        return indices, weights


class LatentDataset_cond(Dataset):
    """Organism-conditional latents only (no potency levels)."""

    def __init__(self, latents_path, organisms_path):
        latents = np.load(latents_path).astype(np.float32)
        organisms = np.load(organisms_path).astype(np.int64)
        if latents.ndim == 3:
            latents = latents.squeeze(1)
        assert len(latents) == len(organisms), (
            f"Length mismatch: {len(latents)} latents vs {len(organisms)} labels"
        )
        latents = latents * 25.0
        self.latents = latents[:, np.newaxis, :]
        self.organisms = organisms
        self.num_organisms = int(organisms.max()) + 1
        print(f"[cond] {len(self.latents)} samples  organisms={self.num_organisms}")
        for i in range(self.num_organisms):
            print(f"  org {i}: {(self.organisms == i).sum()} samples")

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        latent = torch.from_numpy(self.latents[idx])
        org_id = torch.tensor(self.organisms[idx], dtype=torch.long)
        return latent, org_id


# Paths / hyperparams
LATENT_DIR = f"{INPUT_PATH}/latent_two_organism/cond_latents_new"

P2 = dict(
    latents_path=f"{LATENT_DIR}/cond_latents.npy",
    organisms_path=f"{LATENT_DIR}/cond_organisms.npy",
    phase1_ckpt=f"{INPUT_PATH}/checkpoints_diffusion/phase1/best_model.pth",
    save_path=f"{INPUT_PATH}/checkpoints_diffusion/phase2_2org",
    lr=1e-4,
    num_epochs=300,
    batch_size=128,
    num_steps=500,
    num_workers=2,
    save_every=10,
    cfg_dropout=0.10,
)

os.makedirs(P2["save_path"], exist_ok=True)
os.makedirs(f"{P2['save_path']}/model", exist_ok=True)

all_data_p2 = LatentDataset_cond(P2["latents_path"], P2["organisms_path"])
num_organisms = all_data_p2.num_organisms
NULL_LABEL = num_organisms

length = len(all_data_p2)
train_size = int(0.8 * length)
val_size = length - train_size
train_data_p2, val_data_p2 = random_split(
    all_data_p2,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_organisms = all_data_p2.organisms[list(train_data_p2.indices)]
counts = np.bincount(train_organisms, minlength=num_organisms).astype(np.float32)
class_wt = 1.0 / np.where(counts > 0, counts, 1.0)
sample_wts = class_wt[train_organisms]
weighted_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_wts).float(),
    num_samples=len(sample_wts),
    replacement=True,
)

train_loader_p2 = DataLoader(
    train_data_p2,
    batch_size=P2["batch_size"],
    sampler=weighted_sampler,
    num_workers=P2["num_workers"],
    pin_memory=True,
    drop_last=False,
)
val_loader_p2 = DataLoader(
    val_data_p2,
    batch_size=P2["batch_size"] // 10,
    shuffle=False,
    num_workers=P2["num_workers"],
    pin_memory=True,
    drop_last=False,
)
print(
    f"\nP2  Train={train_size}  Val={val_size}  "
    f"Organisms={num_organisms}  NullLabel={NULL_LABEL}  Levels=OFF"
)

run_p2 = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name="phase2_conditional_no_levels",
    group="AMP_diffusion",
    tags=["phase2", "conditional", "CFG", "no-levels"],
    config={
        **P2,
        "phase": 2,
        "model": "DModel-BertEncoder",
        "num_organisms": num_organisms,
        "null_label": NULL_LABEL,
        "use_levels": False,
        "device": str(device),
        "optimizer": "AdamW",
    },
    reinit=True,
)

model_p2 = DModel(num_class=num_organisms + 1, max_length=1).to(device)

if os.path.exists(P2["phase1_ckpt"]):
    p1_ckpt = torch.load(P2["phase1_ckpt"], map_location=device)
    if isinstance(p1_ckpt, dict) and "model_state" in p1_ckpt:
        p1_ckpt = p1_ckpt["model_state"]
    p2_state = model_p2.state_dict()
    to_load = {
        k: v for k, v in p1_ckpt.items() if k in p2_state and v.shape == p2_state[k].shape
    }
    skipped = [k for k in p1_ckpt if k not in to_load]
    model_p2.load_state_dict(to_load, strict=False)
    print(f"Loaded {len(to_load)} tensors from Phase 1. Skipped: {skipped}")
    wandb.run.summary["p2/p1_tensors_loaded"] = len(to_load)
    wandb.run.summary["p2/p1_tensors_skipped"] = len(skipped)
else:
    print("Phase 1 checkpoint not found — training from scratch.")

diff_p2 = create_gaussian_diffusion(P2["num_steps"])
samp_p2 = UniformSampler(P2["num_steps"])
opt_p2 = AdamW(model_p2.parameters(), lr=P2["lr"])

wandb.watch(model_p2, log="gradients", log_freq=200)

log_path_p2 = f"{P2['save_path']}/train.log"
if not os.path.exists(log_path_p2):
    with open(log_path_p2, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")

best_loss_p2 = float("inf")
cfg_dropout = P2["cfg_dropout"]
p2_global_step = 0
p2_smooth = _SmoothBuffer(size=50)
p2_smooth_cfg = _SmoothBuffer(size=50)
p2_smooth_cond = _SmoothBuffer(size=50)

print("\nStarting Phase 2 training (organism-only, no levels)...")

for epoch in range(P2["num_epochs"]):
    model_p2.train()
    train_loss, n_train = 0.0, 0
    cfg_loss_sum, n_cfg = 0.0, 0
    cond_loss_sum, n_cond = 0.0, 0
    start = time.time()
    log_rows = []
    epoch_batch_losses_p2 = []

    for i, (data, organism_ids) in enumerate(
        tqdm(train_loader_p2, desc=f"[P2 Ep {epoch:3d}] train")
    ):
        data = data.to(device)
        organism_ids = organism_ids.to(device)
        drop_mask = torch.rand(organism_ids.shape[0], device=device) < cfg_dropout
        organism_ids = torch.where(
            drop_mask,
            torch.full_like(organism_ids, NULL_LABEL),
            organism_ids,
        )

        opt_p2.zero_grad()
        t, _ = samp_p2.sample(data.shape[0], device)
        loss = diff_p2.cal_loss(model_p2, data, t, condition=organism_ids).mean()
        loss.backward()
        grad_norm = float(
            torch.nn.utils.clip_grad_norm_(model_p2.parameters(), max_norm=1e9)
        )
        opt_p2.step()

        loss_val = loss.item()
        n_dropped = int(drop_mask.sum().item())
        n_kept = len(drop_mask) - n_dropped
        train_loss += loss_val
        n_train += 1

        if n_dropped > 0:
            cfg_loss_sum += loss_val
            n_cfg += 1
            p2_smooth_cfg.push(loss_val)
        if n_kept > 0:
            cond_loss_sum += loss_val
            n_cond += 1
            p2_smooth_cond.push(loss_val)

        p2_smooth.push(loss_val)
        epoch_batch_losses_p2.append(loss_val)
        log_rows.append(f"{epoch},{i},train,{loss_val:.6f},{time.time()-start:.2f}")

        wandb.log(
            {
                "p2/batch/loss_raw": loss_val,
                "p2/batch/loss_smoothed": p2_smooth.mean,
                "p2/batch/loss_std": p2_smooth.std,
                "p2/batch/loss_cfg_smooth": p2_smooth_cfg.mean,
                "p2/batch/loss_cond_smooth": p2_smooth_cond.mean,
                "p2/batch/n_cfg_dropped": n_dropped,
                "p2/batch/n_conditioned": n_kept,
                "p2/batch/grad_norm": grad_norm,
                "p2/batch/lr": opt_p2.param_groups[0]["lr"],
                "p2/step": p2_global_step,
            },
            step=p2_global_step,
        )
        p2_global_step += 1

    train_loss /= n_train

    model_p2.eval()
    val_loss, n_val = 0.0, 0
    val_batch_losses = []
    with torch.no_grad():
        for i, (vdata, organism_ids) in enumerate(
            tqdm(val_loader_p2, desc=f"[P2 Ep {epoch:3d}] val  ")
        ):
            vdata = vdata.to(device)
            organism_ids = organism_ids.to(device)
            t, _ = samp_p2.sample(vdata.shape[0], device)
            loss = diff_p2.cal_loss(model_p2, vdata, t, condition=organism_ids).mean()
            val_loss += loss.item()
            val_batch_losses.append(loss.item())
            n_val += 1
            log_rows.append(f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}")

    val_loss /= n_val
    epoch_time = time.time() - start

    with open(log_path_p2, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    gen_gap = val_loss - train_loss
    avg_cfg_loss = cfg_loss_sum / n_cfg if n_cfg > 0 else float("nan")
    avg_cond_loss = cond_loss_sum / n_cond if n_cond > 0 else float("nan")

    wandb.log(
        {
            "p2/epoch/train_loss": train_loss,
            "p2/epoch/val_loss": val_loss,
            "p2/epoch/generalization_gap": gen_gap,
            "p2/epoch/cfg_dropout_loss": avg_cfg_loss,
            "p2/epoch/cond_loss": avg_cond_loss,
            "p2/epoch/cfg_vs_cond_delta": (
                avg_cfg_loss - avg_cond_loss
                if not (math.isnan(avg_cfg_loss) or math.isnan(avg_cond_loss))
                else float("nan")
            ),
            "p2/epoch/epoch_time_s": epoch_time,
            "p2/epoch/lr": opt_p2.param_groups[0]["lr"],
            "p2/epoch/train_loss_hist": wandb.Histogram(epoch_batch_losses_p2),
            "p2/epoch/val_loss_hist": wandb.Histogram(val_batch_losses),
            "p2/epoch": epoch,
            "p2/step": p2_global_step,
        },
        step=p2_global_step,
    )

    ckpt = {
        "model_state": model_p2.state_dict(),
        "num_organisms": num_organisms,
        "null_label": NULL_LABEL,
        "num_steps": P2["num_steps"],
    }

    if val_loss < best_loss_p2:
        best_loss_p2 = val_loss
        torch.save(ckpt, f"{P2['save_path']}/best_model.pth")
        wandb.run.summary["p2/best_val_loss"] = best_loss_p2
        wandb.run.summary["p2/best_val_epoch"] = epoch
        print(f"  ✓ Best model saved  val={val_loss:.5f}")

    if (epoch + 1) % P2["save_every"] == 0:
        torch.save(
            ckpt,
            f"{P2['save_path']}/model/"
            f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth",
        )

    print(
        f"[P2] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
        f"| gap {gen_gap:+.5f} | cfg {avg_cfg_loss:.5f} | cond {avg_cond_loss:.5f} "
        f"| {epoch_time:.1f}s"
    )

wandb.finish()
print("Phase 2 training complete.")


## Phase 2 - With potency leveling and level embeddings

Organism-conditional diffusion **plus** potency-level conditioning (Weak / Moderate / Strong) via level embeddings and `cond_levels.npy` in the leveled data folder. Starts from the Phase 1 checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import (
    Dataset, DataLoader, WeightedRandomSampler, random_split
)
from tqdm import tqdm
from transformers.models.bert.modeling_bert import BertEncoder, BertConfig

print("Imports OK")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

INPUT_PATH = "/content/drive/MyDrive/DBAASP_Bioinformatics"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")

# Potency level conditioning
LEVEL_NAMES = ["Weak", "Moderate", "Strong"]
LEVEL_NAME_TO_ID = {name.lower(): i for i, name in enumerate(LEVEL_NAMES)}
NULL_LEVEL = len(LEVEL_NAMES)
NUM_LEVEL_EMBEDDINGS = NULL_LEVEL + 1



import wandb

WANDB_PROJECT = "AMP_Generation"
WANDB_ENTITY  = None

class _SmoothBuffer:
    def __init__(self, size=50):
        self._buf  = []
        self._size = size

    def push(self, v):
        self._buf.append(v)
        if len(self._buf) > self._size:
            self._buf.pop(0)

    @property
    def mean(self):
        return float(np.mean(self._buf)) if self._buf else float("nan")

    @property
    def std(self):
        return float(np.std(self._buf))  if len(self._buf) > 1 else 0.0


def timestep_embedding(timesteps, dim, max_period=10000):
    half  = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32)
        / half
    ).to(device=timesteps.device)
    args      = timesteps[:, None].float() * freqs[None]
    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


class DModel(nn.Module):
    def __init__(self, num_class=2, num_levels=0, max_length=1):
        super().__init__()
        self.input_channels = 128
        self.model_channels = 127
        self.out_channels   = 128
        self.num_class      = num_class
        self.num_levels     = num_levels
        self.drop_out       = 0.1
        self.max_length     = max_length
        config = BertConfig()
        config.hidden_dropout_prob     = self.drop_out
        config.max_position_embeddings = self.max_length
        config.num_attention_heads     = 6
        config.num_hidden_layers       = 3
        config._attn_implementation    = "eager"
        self.control_embedding = nn.Embedding(self.num_class, config.hidden_size)
        self.level_embedding = (
            nn.Embedding(num_levels, config.hidden_size) if num_levels > 0 else None
        )
        self.time_embed = nn.Sequential(
            nn.Linear(self.model_channels, config.hidden_size),
            nn.SiLU(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_up_proj = nn.Sequential(
            nn.Linear(self.input_channels, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, config.hidden_size),
        )
        self.input_transformers = BertEncoder(config)
        self.register_buffer(
            "position_ids",
            torch.arange(config.max_position_embeddings).expand((1, -1))
        )
        self.position_embeddings = nn.Embedding(
            config.max_position_embeddings, config.hidden_size
        )
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout   = nn.Dropout(config.hidden_dropout_prob)
        self.output_down_proj = nn.Sequential(
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.Tanh(),
            nn.Linear(config.hidden_size, self.out_channels),
        )

    def forward(self, x, timesteps, control=None, level=None):
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))
        if control is not None:
            emb = emb + self.control_embedding(control)
        if level is not None and self.level_embedding is not None:
            emb = emb + self.level_embedding(level)
        emb_x      = self.input_up_proj(x)
        seq_length = x.size(1)
        pos_ids    = self.position_ids[:, :seq_length]
        emb_inputs = (
            self.position_embeddings(pos_ids)
            + emb_x
            + emb.unsqueeze(1).expand(-1, seq_length, -1)
        )
        emb_inputs = self.dropout(self.LayerNorm(emb_inputs))
        attn_mask  = torch.ones(x.shape[0], seq_length, device=x.device)
        extended_mask = attn_mask[:, None, None, :]
        extended_mask = (1.0 - extended_mask) * torch.finfo(emb_inputs.dtype).min
        h = self.input_transformers(emb_inputs, attention_mask=extended_mask)[0]
        return self.output_down_proj(h).type(x.dtype)


def _extract_into_tensor(arr, timesteps, broadcast_shape):
    res = torch.from_numpy(arr).to(device=timesteps.device)[timesteps].float()
    while len(res.shape) < len(broadcast_shape):
        res = res[..., None]
    return res.expand(broadcast_shape)


def mean_flat(tensor):
    return tensor.mean(dim=list(range(1, len(tensor.shape))))


class MyDiffusion:
    def __init__(self, num_timesteps, betas):
        self.betas         = betas
        alphas             = 1.0 - betas
        self.num_timesteps = num_timesteps
        self.alphas_cumprod      = np.cumprod(alphas, axis=0)
        self.alphas_cumprod_prev = np.append(1.0, self.alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod           = np.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = np.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = (
            betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_log_variance_clipped = np.log(
            np.append(self.posterior_variance[1], self.posterior_variance[1:])
        )
        self.posterior_mean_coef1 = (
            betas * np.sqrt(self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        )
        self.posterior_mean_coef2 = (
            (1.0 - self.alphas_cumprod_prev)
            * np.sqrt(alphas)
            / (1.0 - self.alphas_cumprod)
        )

    def q_sample(self, x_start, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x_start)
        return (
            _extract_into_tensor(self.sqrt_alphas_cumprod, t, x_start.shape) * x_start
            + _extract_into_tensor(self.sqrt_one_minus_alphas_cumprod, t, x_start.shape)
            * noise
        )

    def _scale_timesteps(self, t):
        return t.float() * (1000.0 / self.num_timesteps)

    def get_x_start(self, x_start_mean, std):
        return x_start_mean + std * torch.randn_like(x_start_mean)

    def cal_loss(self, model, x_0, t, condition=None, level=None):
        std = _extract_into_tensor(
            self.sqrt_one_minus_alphas_cumprod,
            torch.tensor([0]).to(x_0.device),
            x_0.shape,
        )
        x_start      = self.get_x_start(x_0, std)
        noise        = torch.randn_like(x_0)
        x_t          = self.q_sample(x_start, t, noise=noise)
        model_output = model(x_t, self._scale_timesteps(t), control=condition, level=level)
        target       = x_start
        mse_loss = mean_flat((target - model_output) ** 2)
        t0_mask  = (t == 0)
        t0_loss  = mean_flat((x_0 - model_output) ** 2)
        return torch.where(t0_mask, t0_loss, mse_loss)

    def q_posterior_mean_variance(self, x_start, x_t, t):
        mean = (
            _extract_into_tensor(self.posterior_mean_coef1, t, x_t.shape) * x_start
            + _extract_into_tensor(self.posterior_mean_coef2, t, x_t.shape) * x_t
        )
        var     = _extract_into_tensor(self.posterior_variance,             t, x_t.shape)
        log_var = _extract_into_tensor(self.posterior_log_variance_clipped, t, x_t.shape)
        return mean, var, log_var

    def p_mean_variance(self, model, x, t, clip_denoised=True, cond=None):
        model_output   = model(x, self._scale_timesteps(t), cond)
        pred_xstart    = model_output.clamp(-1, 1) if clip_denoised else model_output
        model_mean, _, _ = self.q_posterior_mean_variance(pred_xstart, x, t)
        model_log_var  = _extract_into_tensor(
            self.posterior_log_variance_clipped, t, x.shape
        )
        return {"mean": model_mean, "log_variance": model_log_var, "pred_xstart": pred_xstart}

    def p_sample(self, model, x, t, clip_denoised=False, cond=None):
        out          = self.p_mean_variance(model, x, t, clip_denoised, cond)
        noise        = torch.randn_like(x)
        nonzero_mask = (t != 0).float().view(-1, *([1] * (len(x.shape) - 1)))
        sample       = out["mean"] + nonzero_mask * torch.exp(0.5 * out["log_variance"]) * noise
        return {"sample": sample, "pred_xstart": out["pred_xstart"]}

    def p_sample_loop(self, model, shape, cond=None, device="cuda"):
        img = torch.randn(*shape, device=device)
        for i in list(range(self.num_timesteps))[::-1]:
            t = torch.tensor([i] * shape[0], device=device)
            with torch.no_grad():
                out = self.p_sample(model, img, t, clip_denoised=False, cond=cond)
                img = out["sample"]
        return img


def betas_for_alpha_bar(num_diffusion_timesteps, alpha_bar, max_beta=0.999):
    betas = []
    for i in range(num_diffusion_timesteps):
        t1 = i / num_diffusion_timesteps
        t2 = (i + 1) / num_diffusion_timesteps
        betas.append(min(1 - alpha_bar(t2) / alpha_bar(t1), max_beta))
    return np.array(betas)


def create_gaussian_diffusion(steps=500):
    betas = betas_for_alpha_bar(steps, lambda t: 1 - np.sqrt(t + 0.0001))
    return MyDiffusion(num_timesteps=steps, betas=betas)


class UniformSampler:
    def __init__(self, num_timesteps=500):
        self._weights = np.ones([num_timesteps])

    def sample(self, batch_size, device):
        p          = self._weights / np.sum(self._weights)
        indices_np = np.random.choice(len(p), size=(batch_size,), p=p)
        indices    = torch.from_numpy(indices_np).long().to(device)
        weights_np = 1 / (len(p) * p[indices_np])
        weights    = torch.from_numpy(weights_np).float().to(device)
        return indices, weights


class LatentDataset_nocond(Dataset):
    def __init__(self, npy_path):
        raw = np.load(npy_path).astype(np.float32)
        if raw.ndim == 3:
            raw = raw.squeeze(1)
        raw = raw * 25.0
        self.data = raw[:, np.newaxis, :]
        print(f"[nocond] {len(self.data)} samples  shape={self.data.shape[1:]}")

    def __len__(self):  return len(self.data)
    def __getitem__(self, idx): return torch.from_numpy(self.data[idx])


class LatentDataset_cond(Dataset):
    def __init__(self, latents_path, organisms_path, levels_path=None):
        latents   = np.load(latents_path).astype(np.float32)
        organisms = np.load(organisms_path).astype(np.int64)
        if latents.ndim == 3:
            latents = latents.squeeze(1)
        assert len(latents) == len(organisms), (
            f"Length mismatch: {len(latents)} latents vs {len(organisms)} labels"
        )
        latents = latents * 25.0
        self.latents       = latents[:, np.newaxis, :]
        self.organisms     = organisms
        self.num_organisms = int(organisms.max()) + 1
        self.levels        = None
        self.has_levels    = False
        if levels_path and os.path.exists(levels_path):
            levels = np.load(levels_path).astype(np.int64)
            assert len(levels) == len(organisms)
            self.levels = levels
            self.has_levels = True
        level_tag = "on" if self.has_levels else "off"
        print(f"[cond] {len(self.latents)} samples  organisms={self.num_organisms}  levels={level_tag}")
        for i in range(self.num_organisms):
            print(f"  org {i}: {(self.organisms == i).sum()} samples")
        if self.has_levels:
            for i in range(int(self.levels.max()) + 1):
                print(f"  level {i}: {(self.levels == i).sum()} samples")

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        latent = torch.from_numpy(self.latents[idx])
        org_id = torch.tensor(self.organisms[idx], dtype=torch.long)
        if self.has_levels:
            return latent, org_id, torch.tensor(self.levels[idx], dtype=torch.long)
        return latent, org_id


P1 = dict(
    train_latents = f"{INPUT_PATH}/latents_p1/train_latents.npy",
    val_latents   = f"{INPUT_PATH}/latents_p1/val_latents.npy",
    save_path     = f"{INPUT_PATH}/checkpoints_diffusion/phase1",
    lr            = 1e-4,
    num_epochs    = 100,
    batch_size    = 512,
    num_steps     = 500,
    num_workers   = 2,
    save_every    = 10,
)


LATENT_DIR = f"{INPUT_PATH}/latent_withLevelings/2org-baumannii-enterica/cond_latents_new"

P2 = dict(
    latents_path   = f"{LATENT_DIR}/cond_latents.npy",
    organisms_path = f"{LATENT_DIR}/cond_organisms.npy",
    levels_path    = f"{LATENT_DIR}/cond_levels.npy",
    level_dropout  = 0.15,
    phase1_ckpt    = f"{INPUT_PATH}/checkpoints_diffusion/phase1/best_model.pth",
    save_path      = f"{INPUT_PATH}/checkpoints_diffusion/phase2_leveled_2org",
    lr             = 1e-4,
    num_epochs     = 300,
    batch_size     = 128,
    num_steps      = 500,
    num_workers    = 2,
    save_every     = 10,
    cfg_dropout    = 0.10,
)

os.makedirs(P2["save_path"], exist_ok=True)
os.makedirs(f"{P2['save_path']}/model", exist_ok=True)

all_data_p2   = LatentDataset_cond(P2["latents_path"], P2["organisms_path"], P2.get("levels_path"))
num_organisms = all_data_p2.num_organisms
NULL_LABEL    = num_organisms
use_levels    = all_data_p2.has_levels
level_null    = NULL_LEVEL if use_levels else None

length     = len(all_data_p2)
train_size = int(0.8 * length)
val_size   = length - train_size
train_data_p2, val_data_p2 = random_split(
    all_data_p2, [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_organisms = all_data_p2.organisms[list(train_data_p2.indices)]
counts     = np.bincount(train_organisms, minlength=num_organisms).astype(np.float32)
class_wt   = 1.0 / np.where(counts > 0, counts, 1.0)
sample_wts = class_wt[train_organisms]
weighted_sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_wts).float(),
    num_samples=len(sample_wts),
    replacement=True,
)

train_loader_p2 = DataLoader(
    train_data_p2, batch_size=P2["batch_size"],
    sampler=weighted_sampler,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=False,
)
val_loader_p2 = DataLoader(
    val_data_p2, batch_size=P2["batch_size"] // 10,
    shuffle=False,
    num_workers=P2["num_workers"], pin_memory=True, drop_last=False,
)
level_tag = "on" if use_levels else "off"
print(f"\nP2  Train={train_size}  Val={val_size}  "
      f"Organisms={num_organisms}  NullLabel={NULL_LABEL}  "
      f"Levels={level_tag}  LevelNull={level_null}")

run_p2 = wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = "phase2_leveled_9org",
    group   = "AMP_diffusion",
    tags    = ["phase2", "conditional", "CFG", "9-organisms", "leveled"],
    config  = {
        **P2,
        "phase":         2,
        "model":         "DModel-BertEncoder",
        "num_organisms": num_organisms,
        "null_label":    NULL_LABEL,
        "use_levels":    use_levels,
        "level_null":    level_null,
        "num_levels":    NUM_LEVEL_EMBEDDINGS if use_levels else 0,
        "device":        str(device),
        "optimizer":     "AdamW",
    },
    reinit  = True,
)

model_p2 = DModel(
    num_class=num_organisms + 1,
    num_levels=NUM_LEVEL_EMBEDDINGS if use_levels else 0,
    max_length=1,
).to(device)

if os.path.exists(P2["phase1_ckpt"]):
    p1_ckpt = torch.load(P2["phase1_ckpt"], map_location=device)
    if isinstance(p1_ckpt, dict) and "model_state" in p1_ckpt:
        p1_ckpt = p1_ckpt["model_state"]
    p2_state = model_p2.state_dict()
    to_load  = {k: v for k, v in p1_ckpt.items()
                if k in p2_state and v.shape == p2_state[k].shape}
    skipped  = [k for k in p1_ckpt if k not in to_load]
    model_p2.load_state_dict(to_load, strict=False)
    n_loaded = len(to_load)
    print(f"Loaded {n_loaded} tensors from Phase 1. Skipped: {skipped}")
    wandb.run.summary["p2/p1_tensors_loaded"] = n_loaded
    wandb.run.summary["p2/p1_tensors_skipped"] = len(skipped)
else:
    print("Phase 1 checkpoint not found.")

diff_p2 = create_gaussian_diffusion(P2["num_steps"])
samp_p2 = UniformSampler(P2["num_steps"])
opt_p2  = AdamW(model_p2.parameters(), lr=P2["lr"])

wandb.watch(model_p2, log="gradients", log_freq=200)

log_path_p2 = f"{P2['save_path']}/train.log"
if not os.path.exists(log_path_p2):
    with open(log_path_p2, "w") as f:
        f.write("epoch,batch_idx,data_type,loss,run_time\n")

best_loss_p2   = float("inf")
cfg_dropout    = P2["cfg_dropout"]
p2_global_step = 0
p2_smooth      = _SmoothBuffer(size=50)
p2_smooth_cfg  = _SmoothBuffer(size=50)
p2_smooth_cond = _SmoothBuffer(size=50)

print("\nStarting Phase 2 training...")

for epoch in range(P2["num_epochs"]):
    model_p2.train()
    train_loss, n_train    = 0.0, 0
    cfg_loss_sum,  n_cfg   = 0.0, 0
    cond_loss_sum, n_cond  = 0.0, 0
    start    = time.time()
    log_rows = []
    epoch_batch_losses_p2  = []

    for i, batch in enumerate(tqdm(train_loader_p2, desc=f"[P2 Ep {epoch:3d}] train")):
        if use_levels:
            data, organism_ids, level_ids = batch
            level_ids = level_ids.to(device)
        else:
            data, organism_ids = batch
            level_ids = None
        data         = data.to(device)
        organism_ids = organism_ids.to(device)
        drop_mask    = torch.rand(organism_ids.shape[0], device=device) < cfg_dropout
        organism_ids = torch.where(
            drop_mask,
            torch.full_like(organism_ids, NULL_LABEL),
            organism_ids,
        )
        if use_levels and level_ids is not None:
            level_drop = torch.rand(level_ids.shape[0], device=device) < P2.get("level_dropout", 0.15)
            level_ids = torch.where(
                level_drop,
                torch.full_like(level_ids, NULL_LEVEL),
                level_ids,
            )
        opt_p2.zero_grad()
        t, _ = samp_p2.sample(data.shape[0], device)
        loss  = diff_p2.cal_loss(model_p2, data, t, condition=organism_ids, level=level_ids).mean()
        loss.backward()

        grad_norm = float(
            torch.nn.utils.clip_grad_norm_(model_p2.parameters(), max_norm=1e9)
        )

        opt_p2.step()
        loss_val    = loss.item()
        n_dropped   = int(drop_mask.sum().item())
        n_kept      = len(drop_mask) - n_dropped
        train_loss += loss_val
        n_train    += 1

        if n_dropped > 0:
            cfg_loss_sum += loss_val; n_cfg += 1
            p2_smooth_cfg.push(loss_val)
        if n_kept > 0:
            cond_loss_sum += loss_val; n_cond += 1
            p2_smooth_cond.push(loss_val)

        p2_smooth.push(loss_val)
        epoch_batch_losses_p2.append(loss_val)

        log_rows.append(f"{epoch},{i},train,{loss_val:.6f},{time.time()-start:.2f}")

        wandb.log({
            "p2/batch/loss_raw":          loss_val,
            "p2/batch/loss_smoothed":     p2_smooth.mean,
            "p2/batch/loss_std":          p2_smooth.std,
            "p2/batch/loss_cfg_smooth":   p2_smooth_cfg.mean,
            "p2/batch/loss_cond_smooth":  p2_smooth_cond.mean,
            "p2/batch/n_cfg_dropped":     n_dropped,
            "p2/batch/n_conditioned":     n_kept,
            "p2/batch/grad_norm":         grad_norm,
            "p2/batch/lr":                opt_p2.param_groups[0]["lr"],
            "p2/step":                    p2_global_step,
        }, step=p2_global_step)

        p2_global_step += 1

    train_loss /= n_train

    model_p2.eval()
    val_loss, n_val  = 0.0, 0
    val_batch_losses = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(val_loader_p2, desc=f"[P2 Ep {epoch:3d}] val  ")):
            if use_levels:
                vdata, organism_ids, level_ids = batch
                level_ids = level_ids.to(device)
            else:
                vdata, organism_ids = batch
                level_ids = None
            vdata        = vdata.to(device)
            organism_ids = organism_ids.to(device)
            t, _  = samp_p2.sample(vdata.shape[0], device)
            loss  = diff_p2.cal_loss(model_p2, vdata, t, condition=organism_ids, level=level_ids).mean()
            val_loss += loss.item()
            val_batch_losses.append(loss.item())
            n_val    += 1
            log_rows.append(f"{epoch},{i},val,{loss.item():.6f},{time.time()-start:.2f}")

    val_loss /= n_val
    epoch_time = time.time() - start

    with open(log_path_p2, "a") as f:
        f.write("\n".join(log_rows) + "\n")

    gen_gap      = val_loss - train_loss
    avg_cfg_loss  = cfg_loss_sum  / n_cfg  if n_cfg  > 0 else float("nan")
    avg_cond_loss = cond_loss_sum / n_cond if n_cond > 0 else float("nan")

    wandb.log({
        "p2/epoch/train_loss":           train_loss,
        "p2/epoch/val_loss":             val_loss,
        "p2/epoch/generalization_gap":   gen_gap,
        "p2/epoch/cfg_dropout_loss":     avg_cfg_loss,
        "p2/epoch/cond_loss":            avg_cond_loss,
        "p2/epoch/cfg_vs_cond_delta":    avg_cfg_loss - avg_cond_loss
                                         if not (math.isnan(avg_cfg_loss) or
                                                 math.isnan(avg_cond_loss)) else float("nan"),
        "p2/epoch/epoch_time_s":         epoch_time,
        "p2/epoch/lr":                   opt_p2.param_groups[0]["lr"],
        "p2/epoch/train_loss_hist":      wandb.Histogram(epoch_batch_losses_p2),
        "p2/epoch/val_loss_hist":        wandb.Histogram(val_batch_losses),
        "p2/epoch":                      epoch,
        "p2/step":                       p2_global_step,
    }, step=p2_global_step)

    if val_loss < best_loss_p2:
        best_loss_p2 = val_loss
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
                "num_levels":    NUM_LEVEL_EMBEDDINGS if use_levels else 0,
                "level_null":    NULL_LEVEL if use_levels else None,
            },
            f"{P2['save_path']}/best_model.pth",
        )
        wandb.run.summary["p2/best_val_loss"]  = best_loss_p2
        wandb.run.summary["p2/best_val_epoch"] = epoch
        print(f"  ✓ Best model saved  val={val_loss:.5f}")

    if (epoch + 1) % P2["save_every"] == 0:
        torch.save(
            {
                "model_state":   model_p2.state_dict(),
                "num_organisms": num_organisms,
                "null_label":    NULL_LABEL,
                "num_steps":     P2["num_steps"],
                "num_levels":    NUM_LEVEL_EMBEDDINGS if use_levels else 0,
                "level_null":    NULL_LEVEL if use_levels else None,
            },
            f"{P2['save_path']}/model/"
            f"ep{epoch:04d}_tr{train_loss:.5f}_val{val_loss:.5f}.pth",
        )

    print(f"[P2] Ep {epoch:4d} | train {train_loss:.5f} | val {val_loss:.5f} "
          f"| gap {gen_gap:+.5f} | cfg {avg_cfg_loss:.5f} | cond {avg_cond_loss:.5f} "
          f"| {epoch_time:.1f}s")

wandb.finish()
print("Phase 2 training complete.")